# Shortlisting candidate structures from an MS/MS spectrum

This notebook runs the companion tool on three real MassSpecGym spectra chosen by a fixed rule
(the first [M+H]+ and the first [M+Na]+ spectrum of the formula-split test fold, plus the
[M+H]+ spectrum with its true structure removed from the candidate file). It then loads the saved
benchmark receipts behind the article's numbers.

A shortlist is a set of structures to confirm, not an identification. Scores are not probabilities.
Identity is 2D: stereoisomers are not distinguished.

In [1]:
import json, platform, subprocess, sys
from pathlib import Path
import pandas as pd
import torch, rdkit
ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(ROOT))
from msms_shortlist import __version__
from msms_shortlist.chem import exact_mass, neutral_mass, ppm_error
EX, MODELS = ROOT / "examples", ROOT / "models"
DREAMS = Path(ROOT / "weights" / "DreaMS_embedding_model_torchscript.pt")
OUT = ROOT / "notebook" / "outputs"; OUT.mkdir(exist_ok=True)
print("msms_shortlist", __version__, "| python", platform.python_version(), "| torch", torch.__version__, "| rdkit", rdkit.__version__)
print("DreaMS TorchScript present:", DREAMS.exists(), "| model bundle present:", (MODELS / "dreams_morgan_formula_seed1_fp16.pt").exists())

msms_shortlist 0.1.0 | python 3.11.13 | torch 2.2.1 | rdkit 2023.09.6
DreaMS TorchScript present: True | model bundle present: True


## 1. Check the metadata before ranking anything

The neutral mass comes from the precursor m/z, the adduct and the charge. If any of them is missing
or inconsistent, the tool refuses to rank.

In [2]:
demo = json.loads((EX / "demo-inputs.json").read_text())
for key in ("mh", "mna"):
    d = demo[key]
    implied = neutral_mass(d["precursor_mz"], d["adduct"], 1)
    true = exact_mass(d["true_smiles_2d"])
    print(f"{key}: {d['massspecgym_identifier']} {d['adduct']} precursor {d['precursor_mz']} -> neutral {implied:.5f}; "
          f"annotated structure {true:.5f} ({float(ppm_error(implied, true)):+.2f} ppm); {d['n_candidates']} candidates")

mh: MassSpecGymID0226249 [M+H]+ precursor 330.98991 -> neutral 329.98263; annotated structure 329.98263 (+0.01 ppm); 254 candidates
mna: MassSpecGymID0395778 [M+Na]+ precursor 826.4 -> neutral 803.41078; annotated structure 803.50444 (-116.57 ppm); 229 candidates


In [3]:
def run(spectra, candidates, out, model=True, extra=()):
    cmd = [sys.executable, "-m", "msms_shortlist.cli", "--spectra", str(EX / spectra),
           "--candidates", str(EX / candidates), "--out", str(OUT / out), *extra]
    if model:
        cmd += ["--model", str(MODELS / "dreams_morgan_formula_seed1_fp16.pt"), "--dreams", str(DREAMS),
                "--thresholds", str(MODELS / "thresholds.json")]
    print(subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip())
    prov = json.loads((OUT / out).with_suffix(".provenance.json").read_text())
    table = pd.read_csv(OUT / out) if (OUT / out).stat().st_size > 1 else pd.DataFrame()
    return table, prov

run("invalid-metadata.mgf", "mh-candidates.csv", "invalid.csv", model=False);

no_adduct_example: refused (missing ADDUCT; the neutral mass cannot be derived)
wrong_charge_example: refused (charge -1 is inconsistent with adduct [M+H]+)


## 2. The cheap baseline: rank by precursor mass error

Every isomer of the true structure has exactly the same monoisotopic mass, so mass error cannot
separate them. The tool reports how many candidates tie at the shortlist boundary.

In [4]:
mass, prov = run("mh.mgf", "mh-candidates.csv", "mh-mass.csv", model=False)
q = prov["queries"][0]
print("tie at shortlist boundary:", q["tie_at_shortlist_boundary"], "| candidates sharing the 5th score:", q["n_candidates_tied_with_shortlist_boundary"])
mass.head(8)[["rank", "id", "smiles_2d", "ppm_error", "tied_with_previous", "in_shortlist"]]

MassSpecGymID0226249: nominate shortlist
tie at shortlist boundary: True | candidates sharing the 5th score: 30


,rank,id,smiles_2d,ppm_error,tied_with_previous,in_shortlist
0,1,cand070,Brc1ccccc1C1SCc2nc3ccccc3n21,0.006352,False,True
1,2,cand005,Nc1sc(-c2ccccc2)nc1-c1ccc(Br)cc1,0.006352,True,True
2,3,cand251,Nc1ccc(Sc2cccc(Br)c2)c2cnccc12,0.006352,True,True
3,4,cand014,C=Cc1cn(-c2ccc(Br)cc2)nc1-c1cccs1,0.006352,True,True
4,5,cand182,Cc1[nH]c(-c2ccc3ccccc3c2)nc(=S)c1Br,0.006352,True,True
5,6,cand012,CCc1nc2cccc3c2n1-c1ccc(Br)cc1S3,0.006352,True,False
6,7,cand000,Brc1ccc(Nc2nc(-c3ccccc3)cs2)cc1,0.006352,True,False
7,8,cand177,Nc1cccc(-c2nc(-c3cccc(Br)c3)cs2)c1,0.006352,True,False


## 3. The learned model: DreaMS spectrum embedding + Morgan fingerprint alignment

The model was trained here with the released MSAlign code (DreaMS + Morgan pair, released
formula-split settings, one seed). The decline threshold was fitted on validation spectra so that at
most 10% of queries with their answer removed would still get a shortlist.

In [5]:
model, prov = run("mh.mgf", "mh-candidates.csv", "mh-model.csv")
truth = demo["mh"]["true_structure_id"]
print("decision:", prov["queries"][0]["decision"], "| top score:", round(prov["queries"][0]["top_score"], 4),
      "| threshold:", round(prov["thresholds"]["tau"], 4))
print("rank of the annotated structure:", int(model.loc[model.id == truth, "rank"].iloc[0]))
model.head(5)[["rank", "id", "smiles_2d", "score_model", "ppm_error"]]

MassSpecGymID0226249: decline: top score below validation threshold
decision: decline: top score below validation threshold | top score: 0.0194 | threshold: 0.5994
rank of the annotated structure: 1


,rank,id,smiles_2d,score_model,ppm_error
0,1,cand000,Brc1ccc(Nc2nc(-c3ccccc3)cs2)cc1,0.019414,0.006352
1,2,cand181,Cn1nnc(NC(=S)NC(=O)c2cccc(Cl)c2Cl)n1,-0.024529,-9.399461
2,3,cand005,Nc1sc(-c2ccccc2)nc1-c1ccc(Br)cc1,-0.038680,0.006352
3,4,cand139,CCOC(=O)CNC(=O)c1cccc([N+](=O)[O-])c1Br,-0.056382,-7.576111
4,5,cand149,Cn1nnc(NC(=S)NC(=O)c2ccc(Cl)cc2Cl)n1,-0.064109,-9.399461


The true structure is ranked first, yet the tool declines: its top score is far below the
threshold. On the test fold that threshold nominated only about one target-present query in five.
A strict decline rule buys fewer false nominations with a lot of silence.

In [6]:
absent, prov = run("mh.mgf", "mh-candidates-target-removed.csv", "mh-absent-model.csv")
print("decision with the true structure removed:", prov["queries"][0]["decision"],
      "| top score:", round(prov["queries"][0]["top_score"], 4))
absent.head(5)[["rank", "id", "smiles_2d", "score_model"]]

MassSpecGymID0226249: decline: top score below validation threshold
decision with the true structure removed: decline: top score below validation threshold | top score: -0.0245


,rank,id,smiles_2d,score_model
0,1,cand181,Cn1nnc(NC(=S)NC(=O)c2cccc(Cl)c2Cl)n1,-0.024529
1,2,cand005,Nc1sc(-c2ccccc2)nc1-c1ccc(Br)cc1,-0.038680
2,3,cand139,CCOC(=O)CNC(=O)c1cccc([N+](=O)[O-])c1Br,-0.056381
3,4,cand149,Cn1nnc(NC(=S)NC(=O)c2ccc(Cl)cc2Cl)n1,-0.064109
4,5,cand079,Cc1sc(-c2ccncc2)nc1-c1ccc(Br)cc1,-0.068101


## 4. A failure before ranking: the precursor value itself

The [M+Na]+ spectrum selected by the same rule records its precursor as 826.4. The annotated
structure implies 826.494, so the true answer lies 117 ppm away. A 10 or 50 ppm window around the
recorded value contains none of the candidates, and the tool says so instead of ranking an empty pool.
Only a window wider than about 120 ppm brings them back, and against a real database such a window
admits far more structures than this frozen candidate file holds.

In [7]:
for w in ("10", "50", "200"):
    _, prov = run("mna.mgf", "mna-candidates.csv", f"mna-{w}ppm.csv", extra=("--ppm", w))
    print(w, "ppm:", prov["queries"][0]["decision"], "| candidates in window:", prov["queries"][0].get("n_candidates_in_window"))

MassSpecGymID0395778: no candidate within window (no structure within 10.0 ppm)
10 ppm: no candidate within window | candidates in window: 0


MassSpecGymID0395778: no candidate within window (no structure within 50.0 ppm)
50 ppm: no candidate within window | candidates in window: 0


MassSpecGymID0395778: decline: top score below validation threshold
200 ppm: decline: top score below validation threshold | candidates in window: 227


## 5. The benchmark receipts behind the article

These tables are read from the saved analysis outputs; nothing is re-run here.

In [8]:
res = ROOT / "results"
metrics = pd.read_csv(res / "metrics.csv")
view = metrics.query("pool == 'official_dedup' and rule == 'expected'").pivot(index="method", columns="k", values="estimate")
view.round(1)

k,1,5,20
method,,,
deepsets,13.4,31.3,53.9
embcos,36.1,63.2,82.4
fusion,35.2,64.8,86.2
mass,3.7,13.3,32.0
msalign,33.2,61.4,83.9
random,0.4,2.1,8.5


In [9]:
metrics.query("rule == 'expected' and k == 5").pivot(index="method", columns="pool", values="estimate").round(1)

pool,expanded1024,official_dedup,official_raw,sub16,sub64
method,,,,,
deepsets,19.9,31.3,28.9,73.9,51.8
embcos,45.9,63.2,60.4,93.8,80.4
fusion,43.9,64.8,61.2,96.2,84.2
mass,8.2,13.3,12.9,62.5,30.7
msalign,41.3,61.4,57.9,94.5,81.4
random,1.3,2.1,2.0,31.2,7.8


In [10]:
ab = pd.read_csv(res / "abstention.csv")
ab.query("confidence == 'top1'")[["method", "threshold", "coverage_present_estimate",
    "recall5_among_nominated_estimate", "false_nomination_absent_estimate"]].round(3)

,method,threshold,coverage_present_estimate,recall5_among_nominated_estimate,false_nomination_absent_estimate
0,mass,tau_cov90,0.900,0.147,0.900
1,mass,tau_fn10,0.122,0.221,0.121
4,msalign,tau_cov90,0.908,0.643,0.866
5,msalign,tau_fn10,0.213,0.684,0.132
8,embcos,tau_cov90,0.908,0.666,0.856
9,embcos,tau_fn10,0.255,0.837,0.157
12,deepsets,tau_cov90,0.875,0.332,0.861
13,deepsets,tau_fn10,0.184,0.528,0.151
16,fusion,tau_cov90,0.869,0.707,0.838
17,fusion,tau_fn10,0.218,0.868,0.102
